# Matrix chain multiplication


Canonical DP: given a sequence of matrices $M_0, M_1, \ldots, M_{N-1}$
where $M_i$ has shape $d_i \times d_{i+1}$, the number of scalar
multiplications needed depends on the parenthesization. The DP finds
the best one in $O(N^3)$.

- A = $10 \times 30$ matrix
- B = $30 \times 5$ matrix
- C = $5 \times 60$ matrix

- $(AB)C$ costs $(10 \cdot 30 \cdot 5) + (10 \cdot 5 \cdot 60) = 1500 + 3000 = 4500$.
- $A(BC)$ costs $(30 \cdot 5 \cdot 60) + (10 \cdot 30 \cdot 60) = 9000 + 18000 = 27000$.

Reference: [Matrix chain multiplication](https://en.wikipedia.org/wiki/Matrix_chain_multiplication).


## Construction

We import the hypergraph-returning version from the applications module and show its source inline with `arsenal.nb.psource` so this notebook can't drift from the code:

In [1]:
from arsenal.nb import psource
from hypergraphs.apps.matrix_chain import matrix_chain

psource(matrix_chain)


## Running it

Build the forest for dims $[10, 30, 5, 60]$ under MinPlus and read off the minimum number of scalar multiplications:

In [2]:
from semirings import MinPlus

g = matrix_chain([10, 30, 5, 60], MinPlus)
g.Z().cost


4500

The forest itself:

In [3]:
g.display()


## Same forest, different semiring

`g.apply(f)` re-weights each hyperedge. Count the parenthesizations by mapping every edge to $\mathbf{1}$ under the counting semiring:

In [4]:
from semirings import Count

g.apply(lambda e: Count(1)).Z().x    # C_{N-1} = Catalan(2) = 2


2

List every parenthesization with its cost (note: `g.sorted()` is MaxPlus-oriented — the first item is the *worst* parenthesization; see TODO.md):

In [5]:
for item in g.sorted():
    print(item.score.cost, item.data)


27000 [Edge(weight=MinPlus(18000, (0, 2, 0)), head=(0, 2), body=((0, 0), (1, 2))), [Edge(weight=MinPlus(0.0, ()), head=(0, 0), body=()), [Edge(weight=MinPlus(9000, (1, 2, 1)), head=(1, 2), body=((1, 1), (2, 2))), [Edge(weight=MinPlus(0.0, ()), head=(1, 1), body=()), Edge(weight=MinPlus(0.0, ()), head=(2, 2), body=())]]]]
4500 [Edge(weight=MinPlus(3000, (0, 2, 1)), head=(0, 2), body=((0, 1), (2, 2))), [[Edge(weight=MinPlus(1500, (0, 1, 0)), head=(0, 1), body=((0, 0), (1, 1))), [Edge(weight=MinPlus(0.0, ()), head=(0, 0), body=()), Edge(weight=MinPlus(0.0, ()), head=(1, 1), body=())]], Edge(weight=MinPlus(0.0, ()), head=(2, 2), body=())]]


## TODO

Make the backpointer representation more readable — something like `((A*B)*C)` instead of the nested tuple / `Edge` structure.